# Recommended FTIR preprocessing pipeline (with optional Mie scatter correction)

This notebook teaches **the preprocessing pipeline SpecML recommends by default**, one step at a time.

Most users should preprocess by calling a single line:

```python
sd = SpectroscopicData(file_path="...").easy_preprocess()
```

but if you don't know what `easy_preprocess` is doing under the hood, you can't tell when it's the wrong choice. So in this notebook we run each step manually, plot the result, and explain what it's for. At the end we add an **optional ME-EMSC scattering correction** for spectra with significant Mie scattering (single cells, thin tissue cores, distorted amide I shape). We'll then compare the manual pipeline to the one-liner.

What you'll cover:
1. Wavenumber range trimming (950–3550 cm⁻¹)
2. Paraffin/wax and CO₂ region removal
3. Tissue masking via amide I area
4. Min-to-zero baseline
5. Vector normalisation
6. Trim to the fingerprint region (1000–1800 cm⁻¹)
7. (Optional) Savitzky–Golay derivative
8. (Optional, advanced) ME-EMSC Mie scattering correction

Cells you should edit are marked `# EDIT ME`.

## 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from specml.data.spectroscopic_data import SpectroscopicData

%matplotlib inline

def plot_spectra(sd, n=8, title="", ax=None):
    """Plot the mean (black) and a few random pixels (faint colour)."""
    rng = np.random.default_rng(0)
    idx = rng.choice(sd.data.shape[0], size=min(n, sd.data.shape[0]), replace=False)
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 4))
    for i in idx:
        ax.plot(sd.wavenumbers, sd.data[i], lw=0.7, alpha=0.5)
    ax.plot(sd.wavenumbers, sd.data.mean(axis=0), color="k", lw=2, label="mean")
    ax.invert_xaxis()
    ax.set_xlabel("Wavenumber (cm$^{-1}$)")
    ax.set_ylabel("Intensity")
    ax.set_title(title)
    ax.legend(loc="best")
    return ax

## 1. Load your FTIR data

Point at an Agilent FTIR file (`.dmt`, `.dms`, `.dmd`, `.drd`) or a Zarr directory.

In [ ]:
FILE_PATH = "/path/to/your/file.dmt"  # EDIT ME

sd_raw = SpectroscopicData(file_path=FILE_PATH)
print("raw shape       :", sd_raw.data.shape, "(n_pixels, n_wavenumbers)")
print("wavenumber range:", f"{sd_raw.wavenumbers.min():.0f} - {sd_raw.wavenumbers.max():.0f} cm^-1")
print("image size      :", f"{sd_raw.ypixels} x {sd_raw.xpixels} pixels")

plot_spectra(sd_raw, title="raw spectra")
plt.show()

We'll work on a **copy** of the data so you can re-run individual cells without reloading the file. Each step below uses `inplace=False`, returning a new `SpectroscopicData` object.

In [ ]:
sd = SpectroscopicData(file_path=FILE_PATH)
sd.inplace = False  # methods will return a new object instead of mutating sd

## Step 1 — Trim to a broad clean range (950–3550 cm⁻¹)

FTIR detectors record outside the chemically interesting range. Trimming early speeds up everything downstream and avoids edge artefacts in the next steps. The 950–3550 window keeps both the **fingerprint region** (1000–1800) and the **C–H/O–H stretch region** (2700–3500) so we can still identify wax peaks before removing them.

In [ ]:
sd = sd.keep_range(950, 3550)
print("shape after trim:", sd.data.shape)
plot_spectra(sd, title="after keep_range(950, 3550)")
plt.show()

## Step 2 — Remove paraffin (wax) and CO₂ regions

**Paraffin** is used in FFPE sample preparation and contributes peaks that are nothing to do with tissue chemistry. SpecML's `remove_wax()` cuts three windows:
- 3000–2700 cm⁻¹ (C–H stretches),
- 2400–2300 cm⁻¹,
- 1490–1350 cm⁻¹ (C–H bending).

**CO₂** absorbs at ~2350 cm⁻¹. If background correction wasn't perfect during acquisition you'll see a sharp band there — `remove_co2()` excises 2450–2250 cm⁻¹.

These functions don't *fill* the gap — they delete those wavenumbers entirely. The plot will have visible discontinuities, which is fine: the points are no longer adjacent.

In [ ]:
sd = sd.remove_wax().remove_co2()
print("shape after wax+CO2 removal:", sd.data.shape)
plot_spectra(sd, title="after remove_wax + remove_co2")
plt.show()

## Step 3 — Build a tissue mask from the amide I band

Most pixels in a hyperspectral image are **background** (no tissue, just noise). Including them in PCA, clustering, or model training is wasteful and harmful — they dominate the variance and pull cluster centres away from real tissue.

The trick: pixels with real tissue have a strong amide I band (1600–1700 cm⁻¹, the C=O stretch of peptide bonds). We compute the integrated absorbance there for every pixel; pixels above a threshold are kept.

`area_threshold` is the most important knob here. **Tune it to your data** — too high and you lose tissue, too low and you keep noise. Plot the area first to pick a sensible value.

In [ ]:
AREA_RANGE = (1600, 1700)        # amide I — tracks total protein
AREA_THRESHOLD = 1.5             # EDIT ME — depends on your data

amide_area = sd.calculate_area(*AREA_RANGE)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(amide_area, bins=80)
ax[0].axvline(AREA_THRESHOLD, color="red", lw=2, label=f"threshold = {AREA_THRESHOLD}")
ax[0].set_xlabel("integrated amide I area"); ax[0].set_ylabel("pixel count")
ax[0].set_title("pick a threshold above the noise floor"); ax[0].legend()

# image view of the area
area_img = np.full(sd.xpixels * sd.ypixels, np.nan)
area_img[sd.mask] = amide_area
ax[1].imshow(area_img.reshape(sd.ypixels, sd.xpixels), cmap="viridis")
ax[1].set_title("amide I area per pixel")
ax[1].set_xticks([]); ax[1].set_yticks([])
plt.tight_layout(); plt.show()

Once you're happy with the threshold, apply the mask:

In [ ]:
from specml.processing.masking import create_mask_from_area, apply_mask, rebuild_tissue_mask

tissue_mask = create_mask_from_area(area=amide_area, threshold=AREA_THRESHOLD)
sd.data = apply_mask(sd.data, tissue_mask)
sd.mask = rebuild_tissue_mask(sd, tissue_mask)

print(f"kept {sd.data.shape[0]} of {amide_area.size} pixels "
      f"({100 * sd.data.shape[0] / amide_area.size:.1f}%)")
plot_spectra(sd, title="after tissue masking")
plt.show()

## Step 4 — Min-to-zero baseline

Crude but fast: shift every spectrum so its lowest point sits at zero. Removes simple offset baselines without fitting anything.

If you have curved baselines (e.g. heavy Mie scatter), this won't fix them — see step 8 (ME-EMSC) instead.

In [ ]:
sd = sd.adjust_min_to_zero()
plot_spectra(sd, title="after adjust_min_to_zero")
plt.show()

## Step 5 — Vector normalisation

Different pixels have different overall absorbance because tissue is not uniformly thick. Vector normalisation divides each spectrum by its L2 norm so they all sit on the unit hypersphere. Only the **shape** of each spectrum survives — and shape is what carries chemical information.

After this all spectra should look comparable in scale.

In [ ]:
sd = sd.normalize(method="vector")
plot_spectra(sd, title="after vector normalisation")
plt.show()

## Step 6 — Trim to the fingerprint region (1000–1800 cm⁻¹)

The 1000–1800 cm⁻¹ range carries almost all of the chemical information used in tissue classification — amide I, amide II, lipids, nucleic acids, carbohydrates. The C–H stretch region is mostly redundant for classification and brings noise. Most published FTIR ML models are trained on this window.

In [ ]:
sd = sd.keep_range(1000, 1800)
print("final shape:", sd.data.shape)
plot_spectra(sd, title="after fingerprint trim — final preprocessed")
plt.show()

## Step 7 (optional) — Savitzky–Golay derivative

First or second derivatives can help downstream models because they:
- remove constant or linear baselines that survived earlier steps,
- sharpen overlapping bands.

They also amplify noise, so a smoothing-+-derivative pipeline (Savitzky-Golay does both) is the usual approach. Set `DO_DERIVATIVE = True` to try it.

In [ ]:
DO_DERIVATIVE = False  # EDIT ME
DERIV_ORDER = 2        # 1 or 2

if DO_DERIVATIVE:
    sd = sd.apply_derivative(window_length=23, polyorder=5,
                              deriv=DERIV_ORDER, interp=True)
    plot_spectra(sd, title=f"after {DERIV_ORDER}-order Savitzky-Golay derivative")
    plt.show()
else:
    print("skipped")

## Step 8 (optional, advanced) — ME-EMSC Mie scattering correction

**Why this exists.** When the IR wavelength is comparable to the size of cells or tissue features (~2–10 μm), Mie scattering distorts the recorded spectrum. The classic symptom is a curved, sloping baseline and a **dispersive amide I band** — looks like a derivative is mixed into the absorption. Vector normalisation does *not* fix this; it only rescales.

**ME-EMSC** (Konevskikh 2016, BioSpecNorway) builds a basis of Mie extinction curves *Q*ext over a grid of sphere radii and refractive indices, and fits that basis out of each spectrum iteratively. SpecML vendors Dougal Ferguson's MIT-licensed Python port — see `src/specml/processing/me_emsc/`.

**When to use it:**
- single cells, thin tissue cores, anything with visible scatter ripples or distorted band shapes;
- you have (or can compute) a clean reference spectrum (Matrigel, casein, mean of a flat tissue region without scatter).

**When to skip:** bulk tissue with weak scatter — vector norm is enough.

**Caveats:**
- it's iterative and per-spectrum slow. Expect seconds per pixel. For full hyperspectral images (~10⁵ pixels) this is impractical without subsampling.
- use it **instead of** Step 5 (vector norm), not after — ME-EMSC handles the multiplicative scaling itself.
- a real reference matters. The dataset mean is a fallback.

We'll demonstrate it on a small subsample so it finishes in reasonable time.

In [ ]:
DO_ME_EMSC = False  # EDIT ME

if DO_ME_EMSC:
    # Restart from raw data and apply only steps 1-4 + 6 (skip step 5: vector norm),
    # because ME-EMSC handles multiplicative scaling itself.
    sd_me = (
        SpectroscopicData(file_path=FILE_PATH, inplace=False)
        .keep_range(950, 3550)
        .remove_wax()
        .remove_co2()
    )
    # Tissue masking (re-using the threshold from step 3)
    from specml.processing.masking import create_mask_from_area, apply_mask, rebuild_tissue_mask
    area = sd_me.calculate_area(*AREA_RANGE)
    tmask = create_mask_from_area(area=area, threshold=AREA_THRESHOLD)
    sd_me.data = apply_mask(sd_me.data, tmask)
    sd_me.mask = rebuild_tissue_mask(sd_me, tmask)
    sd_me = sd_me.adjust_min_to_zero()
    sd_me = sd_me.keep_range(1000, 1800)

    # Subsample to keep runtime sane.
    n_subsample = min(500, sd_me.data.shape[0])  # EDIT ME — increase if you have time
    rng = np.random.default_rng(0)
    sub_idx = rng.choice(sd_me.data.shape[0], size=n_subsample, replace=False)
    sd_me.data = sd_me.data[sub_idx]

    # Apply ME-EMSC. With reference=None it uses the dataset mean — pass a clean
    # reference spectrum (e.g. Matrigel) here if you have one.
    sd_me = sd_me.apply_me_emsc(
        reference=None,
        options={
            'maxIterationNumber': 30,
            'PCnumber': 8,
            'Weights': True,
        },
        return_residuals=True,
    )

    fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
    plot_spectra(sd_me, title="after ME-EMSC (subsample)", ax=ax[0])
    ax[1].hist(sd_me.me_emsc_rmse, bins=40)
    ax[1].set_xlabel("per-spectrum RMSE"); ax[1].set_ylabel("count")
    ax[1].set_title("ME-EMSC residuals (drop the high tail)")
    plt.tight_layout(); plt.show()
else:
    print("skipped (set DO_ME_EMSC = True to try it)")

## The same thing in one line: `easy_preprocess()`

Once you understand what each step does, you don't need to type them out every time. `easy_preprocess` runs the whole default pipeline (steps 1–6, plus an optional derivative). The result should match what you got at the end of step 6 above.

In [ ]:
sd_easy = SpectroscopicData(file_path=FILE_PATH).easy_preprocess(
    wavenumber_range=(1000, 1800),
    area_range=AREA_RANGE,
    area_threshold=AREA_THRESHOLD,
    deriv=0,
)
print("easy_preprocess output shape:", sd_easy.data.shape)
plot_spectra(sd_easy, title="easy_preprocess() — same pipeline in one line")
plt.show()

## Decision guide: which pipeline should you use?

| Symptom in your raw data                           | Use this                                |
|----------------------------------------------------|------------------------------------------|
| Bulk tissue, clean spectra, just need fast prep    | `easy_preprocess()` (steps 1–6)         |
| Heavy noise                                        | `easy_preprocess()` then `apply_derivative()` (set `deriv=1`) |
| Curved sloping baseline, dispersive amide I        | Steps 1–4, 6, **then** `apply_me_emsc()` |
| Single cells, thin cores                           | Same as above — ME-EMSC                 |
| Different tissues / batches you want to compare    | Build a *common* reference spectrum and use it for ME-EMSC across all of them |

## Try this next

1. Re-run with a different `AREA_THRESHOLD` and watch tissue masking change.
2. Compare `normalize(method="vector")` to `normalize(method="z_score")` at step 5.
3. Turn on `DO_ME_EMSC` with a real reference spectrum (load a Matrigel `.npy` file and pass it as `reference=`).
4. Use `sd.display(1600, 1700, color_bar=True)` at any stage to see the tissue image.
5. Move on to `03_dimensionality_reduction.ipynb` to feed your preprocessed data into PCA/UMAP.